# Suite CON — Constraints

A constraint is a named Basic rule about the instances of one schema, kept beside the schemas; a set gathers
constraints. Both are data: bound classes of their meta-schemas, stored and read back with their rules.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, register } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Validators as SV } from "@mbse/schemas/Framework";
import { assert, raises, same } from "./support.js";

const self = E.variable("this");
const adult = new C.Constraint("Contact", "adult", self.age.ge(18n), "of age");
const named = new C.Constraint("Contact", "named", self.has("name"));
const numbered = new C.Constraint("Phone", "numbered", self.has("number"));

## CON-01 · A constraint is a named rule about one schema's instances

In [ ]:
{
  assert(adult.schema === "Contact" && adult.name === "adult" && adult.description === "of age");
  assert(adult.rule instanceof E.OfOperation.Data && named.description === null); // a writer is resolved to its data
  assert(new C.Constraint("Contact", "same", adult.rule).rule === adult.rule); // data is taken as it is
  assert(adult.schema_name() === "Patterns.Constraint" && adult.owner() === null && adult.identity() === adult.identity());
  assert(adult.validate().length === 0 && new C.Constraint("Contact", "late", null).rule === null);
}

## CON-02 · Constraints are checked statically

In [ ]:
{
  const bad = new C.Constraint("Contact", "x", E.operation("pow", self.age, E.variable("n")));
  assert(same(bad.validate(), ["constraint 'x' of 'Contact': 'pow' is not a core operation",
    "constraint 'x' of 'Contact': argument 1: variable 'n' is not bound"]));
  assert(same(new C.Constraint("", "", null).validate(), ["constraint '' of '': a constraint needs a schema",
    "constraint '' of '': a constraint needs a name", "constraint '' of '': a constraint needs a rule"]));
  const twice = new C.Set([adult, named, new C.Constraint("Contact", "adult", self.age.ge(21n))]);
  assert(same(twice.validate(), ["constraint 'adult' of 'Contact': defined twice"]));
  raises(Errors.ValueError, () => C.check([bad]), "'pow' is not a core operation; constraint 'x' of 'Contact': argument 1");
  const constraints = C.check([adult, named, numbered]); // a set, from any constraints
  assert(constraints instanceof C.Set && C.check(constraints) === constraints);
}

## CON-03 · A set orders constraints and gives each schema's

In [ ]:
{
  const constraints = new C.Set([adult, numbered, named]);
  assert(same(constraints.constraints, [adult, numbered, named]) && Object.isFrozen(constraints.constraints)); // in order
  assert(same(constraints.of("Contact"), [adult, named]) && same(constraints.of("Phone"), [numbered]) && constraints.of("X").length === 0);
  assert(new C.Set().constraints.length === 0 && constraints.schema_name() === "Patterns.Set" && constraints.owner() === null);
  assert(constraints.identity() === constraints.identity() && constraints.identity() !== new C.Set().identity());
}

## CON-04 · A set is data, read back with its rules

In [ ]:
{
  const shared = self.age.ge(18n).data;
  const constraints = new C.Set([new C.Constraint("Contact", "adult", shared, "of age"),
    new C.Constraint("Contact", "phoned", E.operation("implies", shared, self.has("phones")))]);
  const text = SchemaJSON.ToJSON(C.Builders).Reachable(C.Set.Schema, constraints);
  assert(text.split('"kind": "operation"').length - 1 === 4); // get, ge, implies and has: the shared rule's two are written once
  const copy = SchemaJSON.FromJSON(C.Builders).Reachable(C.Set.Schema, text) as C.Set;
  assert(JSON.stringify(copy.constraints.map((c) => [c.schema, c.name, c.description]))
    === JSON.stringify([["Contact", "adult", "of age"], ["Contact", "phoned", null]]));
  assert(Terms.same(copy.constraints[1]!.rule, constraints.constraints[1]!.rule));
  assert((copy.constraints[1]!.rule as any).arguments[0] === copy.constraints[0]!.rule); // still shared
  assert(SV.Validate(C.Builders).Reachable(C.Set.Schema, constraints).length === 0); // valid data, by mbse-schemas' rules
  const built = C.Builders.builder("Patterns.Constraint", adult);
  built.property("name", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("grown"))));
  assert(built.update() === adult && adult.name === "grown" && adult.rule !== null);
  adult.name = "adult";
}

## CON-05 · Decoding refuses what a constraint or a set cannot hold

In [ ]:
{
  const index = (e: any, i: bigint) => e.property("index", (p: any) => p.value((v: any) => v.as_native((n: any) => n.set(i))));
  const ruleEntry = (target: unknown, i = 0n) => (a: any) => a.add((e: any) => {
    if (target !== null) e.link("argument", (l: any) => l.set(target));
    index(e, i);
  });
  const memberEntry = (target: unknown, i: bigint | null = null) => (a: any) => a.add((e: any) => {
    if (target !== null) e.link("constraint", (l: any) => l.set(target));
    if (i !== null) index(e, i);
  });
  raises(Errors.ValueError, () => C.Builders.builder("Patterns.Constraint").adjacency("rule", ruleEntry(self.data)).adjacency(
    "rule", ruleEntry(self.data, 1n)).create(), "a constraint has one rule, got 2");
  raises(TypeError, () => C.Builders.builder("Patterns.Constraint").adjacency("rule", ruleEntry(named)).create(),
    "a rule must be a Basic expression");
  raises(Errors.ValueError, () => C.Builders.builder("Patterns.Constraint").adjacency("rule", ruleEntry(null)).create(),
    "link 'argument' is not set");
  raises(TypeError, () => C.Builders.builder("Patterns.Set").adjacency("constraints", memberEntry(self.data, 0n)).create(),
    "a member must be a constraint");
  raises(Errors.ValueError, () => C.Builders.builder("Patterns.Set").adjacency("constraints", memberEntry(null, 0n)).create(),
    "link 'constraint' is not set");
  const unordered = C.Builders.builder("Patterns.Set").adjacency("constraints", memberEntry(named)).adjacency(
    "constraints", memberEntry(adult, 0n)).adjacency("constraints", memberEntry(numbered)).create() as C.Set;
  assert(same(unordered.constraints, [adult, named, numbered])); // entries without an index come last, in their order
  const blank = C.Builders.builder("Patterns.Constraint").create() as C.Constraint; // what a snapshot leaves out is empty
  assert(blank.schema === "" && blank.name === "" && blank.rule === null && blank.description === null);
}

## CON-06 · Another store holds constraints as proxies

In [ ]:
{
  const store = register(new Proxies.OfStore());
  for (const name of ["Patterns.Constraint", "Patterns.Set", "Patterns.Members", "Expressions.OfOperation"]) {
    assert(store.names().includes(name), name);
  }
  assert(register(store) === store); // what it holds already is skipped
  const proxy = SchemaJSON.FromJSON(store).Reachable(C.Set.Schema,
    SchemaJSON.ToJSON(C.Builders).Reachable(C.Set.Schema, new C.Set([adult]))) as any;
  assert(proxy.schema_name() === "Patterns.Set" && Proxies.store_of(proxy) === store);
}